# 037 — Flujo supervisado y partición train-validation-test

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Aprendizaje supervisado:** dado $D = \{(x_i, y_i)\}$ muestreado i.i.d. de $P(X,Y)$,
buscar $f$ que minimice el riesgo esperado $R(f) = E[L(f(X),Y)]$. Solo podemos medir el
riesgo empírico $\hat{R}$ sobre la muestra; la brecha $R - \hat{R}$ es la generalización.

**Tres particiones, tres funciones:**

- **Train** ajusta parámetros.
- **Validation** compara candidatos/hiperparámetros — su métrica se vuelve optimista con cada elección.
- **Test** se mira UNA sola vez al final: es la única estimación honesta del riesgo real.

**Fuga de datos:** cualquier información de evaluación que llega al entrenamiento
(duplicados, preprocesado ajustado con todo el dataset, fuga temporal, columnas
consecuencia del target). **Baseline:** el modelo trivial (clase mayoritaria, media)
que da denominador a toda métrica.


### 🔁 Flujo que sigue el laboratorio

```text
1. Fijar métrica, split y semilla     4. Comparar candidatos en desarrollo
2. Sellar el test                     5. Elegir UN modelo final
3. Entrenar candidatos con train      6. Medir una vez en test y reportar límites
```

El laboratorio `ml` genera candidatos de umbral y selecciona el de mejor accuracy de
desarrollo. Nota la limitación declarada en su salida: usa el mismo conjunto para
ilustrar la selección — exactamente el sesgo optimista que esta clase enseña a evitar.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Contrato del laboratorio.** Ejecuta `run_lab("ml", seed=37)` en la celda
siguiente y verifica que el resultado incluya `kind`, `evidence` y `limitations`. Anota qué
umbral selecciona y qué accuracy reporta. ¿Contra qué conjunto se midió esa accuracy según
las `limitations`?

**Ejercicio 2 — Split a mano.** Tienes 10 ejemplos con etiquetas
`[L, L, L, L, L, S, S, S, S, S]` y valores de la feature `[0, 1, 1, 2, 3, 3, 4, 5, 6, 7]`
(en ese orden). Haz a mano un split 6/2/2 tomando como test los índices `{3, 9}` y como
validación `{1, 5}`. Para los umbrales `t ∈ {2, 4}` (predecir S si valor ≥ t), calcula el
accuracy en train y en validación, elige el mejor y evalúalo en test. ¿Coincide el ranking
de train con el de validación?

**Ejercicio 3 — Detectar la fuga.** Un equipo normaliza las 10 columnas con media y
desviación del dataset completo, luego hace el split, entrena y reporta test accuracy 0.97.
Explica en 2-3 líneas por qué la cifra está contaminada, qué pasos deben reordenarse y si
esperas que la cifra honesta sea mayor o menor.

**Ejercicio 4 — Semilla y determinismo.** Ejecuta el laboratorio con `seed=37` y
`seed=137` y compara los resultados completos. ¿Qué cambia y qué no? Deduce qué hace la
semilla en este runner en particular, y explica qué DEBERÍA controlar la semilla en un
experimento real con re-muestreo (split, bootstrap) y por qué en ese caso una métrica
reportada con una sola semilla es insuficiente.


In [ ]:
# TODO: ejecuta run_lab("ml", seed=37)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 2: split y selección a mano
valores    = [0, 1, 1, 2, 3, 3, 4, 5, 6, 7]
etiquetas  = ["L", "L", "L", "L", "L", "S", "S", "S", "S", "S"]
idx_test   = {3, 9}
idx_val    = {1, 5}
idx_train  = None  # completa: el resto de los índices

def accuracy_umbral(t, indices):
    # completa: proporción de aciertos prediciendo "S" si valores[i] >= t
    return None

# acc_train_t2, acc_val_t2, acc_train_t4, acc_val_t4, acc_test del elegido


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: dos semillas
r_a = None  # run_lab("ml", seed=37)
r_b = None  # run_lab("ml", seed=137)
# compara los dos dicts completos: ¿qué clave difiere?
# ¿qué concluyes sobre el papel de la semilla en ESTE runner?


## Reflexión

1. El laboratorio selecciona el umbral con el mismo conjunto en el que mide accuracy y lo
   declara en `limitations`. ¿En qué dirección está sesgada la accuracy reportada (1.00) y
   qué partición adicional haría honesta la estimación?
2. Si normalizas los datos con la media de TODO el dataset antes del split, ¿qué tipo de
   fuga cometes y por qué el test deja de estimar el riesgo real aunque el modelo nunca
   haya visto esas filas?
3. Tu modelo logra accuracy 0.93 y la clase mayoritaria es el 92 % de los casos. ¿Qué
   baseline reportarías y qué conclusión honesta admite esa diferencia de 0.01?
